## T5模型完成文本摘要

In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM,DataCollatorForSeq2Seq,Seq2SeqTrainer,Seq2SeqTrainingArguments
import torch
from datasets import load_dataset

In [16]:
ds = load_dataset("supremezxc/nlpcc_2017",split= "train[:1000]")
ds

Dataset({
    features: ['version', 'data'],
    num_rows: 1000
})

In [19]:
ds = ds.remove_columns("version")

In [21]:
ds['data']

[{'content': '四海网讯,近日,有媒体报道称:章子怡真怀孕了!报道还援引知情人士消息称,“章子怡怀孕大概四五个月,预产期是年底前后,现在已经不接工作了。”这到底是怎么回事?消息是真是假?针对此消息,23日晚8时30分,华西都市报记者迅速联系上了与章子怡家里关系极好的知情人士,这位人士向华西都市报记者证实说:“子怡这次确实怀孕了。她已经36岁了,也该怀孕了。章子怡怀上汪峰的孩子后,子怡的父母亲十分高兴。子怡的母亲,已开始悉心照料女儿了。子怡的预产期大概是今年12月底。”当晚9时,华西都市报记者为了求证章子怡怀孕消息,又电话联系章子怡的亲哥哥章子男,但电话通了,一直没有人接听。有关章子怡怀孕的新闻自从2013年9月份章子怡和汪峰恋情以来,就被传N遍了!不过,时间跨入2015年,事情却发生着微妙的变化。2015年3月21日,章子怡担任制片人的电影《从天儿降》开机,在开机发布会上几张合影,让网友又燃起了好奇心:“章子怡真的怀孕了吗?”但后据证实,章子怡的“大肚照”只是影片宣传的噱头。过了四个月的7月22日,《太平轮》新一轮宣传,章子怡又被发现状态不佳,不时深呼吸,不自觉想捂住肚子,又觉得不妥。然后在8月的一天,章子怡和朋友吃饭,在酒店门口被风行工作室拍到了,疑似有孕在身!今年7月11日,汪峰本来在上海要举行演唱会,后来因为台风“灿鸿”取消了。而消息人士称,汪峰原来打算在演唱会上当着章子怡的面宣布重大消息,而且章子怡已经赴上海准备参加演唱会了,怎知遇到台风,只好延期,相信9月26日的演唱会应该还会有惊喜大白天下吧。',
  'title': '知情人透露章子怡怀孕后,父母很高兴。章母已开始悉心照料。据悉,预产期大概是12月底'},
 {'content': '中新社西宁11月22日电(赵凛松)青海省林业厅野生动植物和自然保护区管理局高级工程师张毓22日向中新社记者确认:“经过中国林业科学院、中科院新疆生态与地理研究所和青海省林业厅的共同认定,出现在青海省海西州境内的三只体型较大的鸟为世界极度濒危的红鹳目红鹳科红鹳属的大红鹳。”11月18日,青海省海西州可鲁克湖—托素湖国家级陆生野生动物疫源疫病监测站在野外监测巡护过程中,在可鲁克湖西南岸入水口盐沼滩发现三只体型较大的鸟类。张毓说:“此前在该区域从未发现过这种体型的鸟类。”可鲁克湖—托素湖位于青海省柴达木盆地东北部,

In [23]:
ds = ds.train_test_split(50,seed= 42)
ds

DatasetDict({
    train: Dataset({
        features: ['data'],
        num_rows: 950
    })
    test: Dataset({
        features: ['data'],
        num_rows: 50
    })
})

In [24]:
tokenizer = AutoTokenizer.from_pretrained("Langboat/mengzi-t5-base")
tokenizer

c:\Users\32721\anaconda3\envs\transformers\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\32721\.cache\huggingface\hub\models--Langboat--mengzi-t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tok

T5TokenizerFast(name_or_path='Langboat/mengzi-t5-base', vocab_size=32128, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'additional_special_tokens': ['<extra_id_0>', '<extra_id_1>', '<extra_id_2>', '<extra_id_3>', '<extra_id_4>', '<extra_id_5>', '<extra_id_6>', '<extra_id_7>', '<extra_id_8>', '<extra_id_9>', '<extra_id_10>', '<extra_id_11>', '<extra_id_12>', '<extra_id_13>', '<extra_id_14>', '<extra_id_15>', '<extra_id_16>', '<extra_id_17>', '<extra_id_18>', '<extra_id_19>', '<extra_id_20>', '<extra_id_21>', '<extra_id_22>', '<extra_id_23>', '<extra_id_24>', '<extra_id_25>', '<extra_id_26>', '<extra_id_27>', '<extra_id_28>', '<extra_id_29>', '<extra_id_30>', '<extra_id_31>', '<extra_id_32>', '<extra_id_33>', '<extra_id_34>', '<extra_id_35>', '<extra_id_36>', '<extra_id_37>', '<extra_id_38>', '<extra_id_39>', '<extra_id_40>', '<extra_id_41>', 

In [26]:
def process_data(examples):
    contents = ["摘要生成：\n" + e['content'] for e in examples['data']]
    inputs = tokenizer(contents, max_length= 384 , truncation= True)
    raw_labels = [e['title'] for e in examples['data']]
    labels = tokenizer(text_target= raw_labels, max_length= 64, truncation= True)
    inputs['labels'] = labels["input_ids"]
    return inputs

In [27]:
tokenized_ds = ds.map(process_data,batched= True)
tokenized_ds

Map: 100%|██████████| 50/50 [00:00<00:00, 1384.90 examples/s]


DatasetDict({
    train: Dataset({
        features: ['data', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 950
    })
    test: Dataset({
        features: ['data', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 50
    })
})

In [28]:
tokenizer.decode(tokenized_ds['train'][0]["input_ids"])

'摘要生成: 事故车被路边护栏刺穿(图片来源网络)前天下午,江津区311省道上,一辆载有4人的大众途观轿车突然冲向路边护栏。一段约20米长的护栏从车头穿入,后档玻璃穿出。坐在副驾驶位、中年女司机的母亲不幸遇难。20米护栏刺穿途观据江津相关方面通报称,8月18日16时许,王女士驾车从西湖往李市方向行驶,车辆撞上道路右侧波形护栏。昨天中午记者现场看到,路边4根防护栏固定桩弯曲变形。目击者说,固定桩都是被事故车撞倒的,而护栏刺穿途观令人胆战心惊———目击者提供的现场照片显示,一段约20米长的护栏从途观车标旁穿进车头,贯穿驾驶室后,从后档玻璃穿出,然后高高翘在空中。刺穿途观的护栏已被切割下来。护栏端头原本呈圆形,但此时已变成锲形。现场看不到明显的刹车痕迹。母亲遇难女司机大哭目击者杨先生称,前日下午4点左右,他从江津李市回贾嗣。途经桥土湾附近时,一辆渝A开头的白色大众途观车,像穿糖葫芦一样被贯穿在路边护栏上。车头,一名30多岁的女子正哭得死去活来。他下车了解到,大哭女子姓王,江津贾嗣人。刚才因为开车时分心,这辆才买了几个月的途观车撞上护栏。原来以为就是普通的碰撞,那想一连串撞击下来,坐副驾驶位的母亲居然遇难了。杨先生说,事发时车内有4人。据王女士称,其中两人是她父母。另外一人王</s>'

In [29]:
tokenizer.decode(tokenized_ds['train'][0]["labels"])

'重庆女司机驾车时扭头递手机,护栏刺穿车身致其母身亡,现场无刹车痕迹,据悉女司机有多年驾龄</s>'

In [30]:
model = AutoModelForSeq2SeqLM.from_pretrained("Langboat/mengzi-t5-base")

## 创建评估函数

In [31]:
import numpy as np
from rouge_chinese import Rouge

rouge = Rouge()

def compute_metric(evalpred):
    pred,labels = evalpred
    decode_preds = tokenizer.batch_decode(pred,skip_special_tokens= True)
    labels = np.where(labels != -100, labels , tokenizer.pad_token_id)
    decode_labels = tokenizer.batch_decode(labels,skip_special_tokens=True)
    prediction = [" ".join(p) for p in decode_preds]
    reference = [" ".join(l) for l in decode_labels]
    scores = rouge.get_scores(prediction,reference,avg= True)
    return {
        "rouge-1" : scores["rouge-1"]["f"],
        "rouge-2" : scores["rouge-2"]["f"],
        "rouge-l" : scores["rouge-l"]["f"]
    }
    

In [32]:
args = Seq2SeqTrainingArguments(
    output_dir= "./t5",
    per_device_train_batch_size= 4,
    per_device_eval_batch_size= 8,
    gradient_accumulation_steps= 8,
    logging_steps= 8,
    eval_strategy= "epoch",
    save_strategy= "epoch",
    metric_for_best_model= "rouge-l",
    predict_with_generate= True  ## true才能生成相应来训练
)

In [33]:
trainer = Seq2SeqTrainer(args= args, model= model, train_dataset= tokenized_ds["train"],eval_dataset= tokenized_ds["test"],
                         compute_metrics=compute_metric, tokenizer=tokenizer , data_collator= DataCollatorForSeq2Seq(tokenizer=tokenizer))

C:\Users\32721\AppData\Local\Temp\ipykernel_14448\1071907361.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(args= args, model= model, train_dataset= tokenized_ds["train"],eval_dataset= tokenized_ds["test"],


In [34]:
trainer.train()

c:\Users\32721\anaconda3\envs\transformers\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss,Rouge-1,Rouge-2,Rouge-l
1,3.573000,2.915857,0.410745,0.244319,0.319010
2,2.810600,2.763024,0.415621,0.247232,0.332517


c:\Users\32721\anaconda3\envs\transformers\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\32721\anaconda3\envs\transformers\lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=87, training_loss=3.4115565289026017, metrics={'train_runtime': 5590.0061, 'train_samples_per_second': 0.51, 'train_steps_per_second': 0.016, 'total_flos': 1415088516501504.0, 'train_loss': 3.4115565289026017, 'epoch': 2.907563025210084})

In [35]:
from transformers import pipeline

In [36]:
pipe = pipeline("text2text-generation",model = model,tokenizer= tokenizer)

Device set to use cpu


In [37]:
pipe("摘要生成：\n"+ ds["test"]["data"][-1]["content"],max_length = 64, do_sample = True)

[{'generated_text': '溆浦县发布暴雨橙色预警:预计溆浦县未来3小时降雨量将达50毫米以上,请注意防范,请注意防范。'}]

In [38]:
ds["test"]["data"][-1]["content"]

'发布日期:2015-07-0212:14:00怀化市气象台7月2日12时14分发布暴雨橙色预警信号:预计溆浦县未来3小时降雨量将达50毫米以上,请注意防范。图例标准防御指南3小时内降雨量将达50毫米以上,或者已达50毫米以上且降雨可能持续。1、政府及相关部门按照职责做好防暴雨应急工作;2、切断有危险的室外电源,暂停户外作业;3、处于危险地带的单位应当停课、停业,采取专门措施保护已到校学生、幼儿和其他上班人员的安全;4、做好城市、农田的排涝,注意防范可能引发的山洪、滑坡、泥石流等灾害'